# Notebook 04 — LLM Prompt Engineering via OpenRouter
## Generation de Rapports Medicaux Structures (JSON)

Ce notebook utilise l'API OpenRouter avec un failover sur modeles gratuits (`:free`)
pour eviter les couts et les rate limits.
1. Generer des rapports medicaux structures (JSON valide) sans etre bloque par les limites de taux (rate limits).
2. Iterer sur 3 versions de prompts (simple → structure → chain-of-thought)
3. Comparer la qualite, coherence et fiabilite du parsing JSON
4. Combiner les resultats des notebooks 02 (MLP) et 03 (HuggingFace)

## 1. Imports & Configuration

In [37]:
from __future__ import annotations
import json, os, re, time, warnings
warnings.filterwarnings("ignore")
from pathlib import Path
from getpass import getpass

import pandas as pd
import requests
from dotenv import load_dotenv

base_dir = Path.cwd().resolve()
project_dir = base_dir.parent if base_dir.name == "notebooks" else base_dir
ENV_PATH = project_dir.parent / ".env"
MODELS_DIR = project_dir / "saved_models"

load_dotenv(ENV_PATH)
API_KEY = os.getenv("OPENROUTER_API_KEY", "")

if not API_KEY or not API_KEY.startswith("sk-or-"):
    print("Cle introuvable dans .env — saisie manuelle :")
    API_KEY = getpass("OPENROUTER_API_KEY : ")

print(f"Cle chargee : {API_KEY[:18]}...{API_KEY[-4:]}")

Cle chargee : sk-or-v1-6f7a3147c...4d89


## 2. Client OpenRouter

In [38]:
PRIMARY_MODEL = "openai/gpt-oss-20b:free"
MODEL_CANDIDATES = [
    "openai/gpt-oss-20b:free",
    "mistralai/mistral-7b-instruct:free",
    "google/gemma-4-31b-it:free",
    "deepseek/deepseek-v4-flash:free",
]

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

def call_llm(messages, model=None, temperature=0.3, max_tokens=800, timeout=60, max_retries=2, retry_delay=5.0):
    if model is None:
        model = PRIMARY_MODEL
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "https://github.com/student/diabetes-backend",
        "X-Title": "Diabetes Risk AI Backend"
    }
    payload = {"model": model, "messages": messages, "temperature": temperature, "max_tokens": max_tokens}
    
    last_err = None
    for attempt in range(max_retries + 1):
        t0 = time.time()
        try:
            resp = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=timeout)
            if resp.ok:
                data = resp.json()
                return {"success": True, "content": data["choices"][0]["message"]["content"], "model": data.get("model", model), "latency_ms": int((time.time()-t0)*1000), "tokens": data.get("usage", {})}
            if resp.status_code == 429 and attempt < max_retries:
                time.sleep(retry_delay * (2**attempt))
                continue
            return {"success": False, "content": "", "model": model, "latency_ms": int((time.time()-t0)*1000), "error": f"HTTP {resp.status_code}: {resp.text}"}
        except Exception as e:
            last_err = str(e)
            if attempt < max_retries:
                time.sleep(retry_delay * (2**attempt))
                continue
            return {"success": False, "content": "", "model": model, "latency_ms": int((time.time()-t0)*1000), "error": str(e)}
    return {"success": False, "error": last_err}

def call_llm_failover(messages, max_tokens=800):
    attempts = []
    for m in list(dict.fromkeys(MODEL_CANDIDATES)):
        res = call_llm(messages, model=m, max_tokens=max_tokens)
        attempts.append({"model": m, "success": res.get("success", False), "error": res.get("error")})
        if res.get("success"):
            res["attempts"] = attempts
            return res
        print(f"Echec {m} : {res.get('error')}")
    return {"success": False, "content": "", "model": "local_fallback", "latency_ms": 0, "error": "All failed", "attempts": attempts}

def extract_json(text):
    try: return json.loads(text)
    except: pass
    match = re.search(r"```(?:json)?\s*({.*?})\s*```", text, re.DOTALL)
    if match:
        try: return json.loads(match.group(1))
        except: pass
    match2 = re.search(r"({[^{}]*(?:{[^{}]*}[^{}]*)*})", text, re.DOTALL)
    if match2:
        try: return json.loads(match2.group(1))
        except: pass
    return None

def local_fallback(pat):
    return {"risk_level": "moderate", "summary": "Mode local (API indisponible).", "recommendations": ["Consulter un medecin"], "urgency": "soon", "generation_mode": "local_fallback"}

## 3. Test de connexion

In [39]:
test_msg = [{"role": "user", "content": "Reply with exactly: OK"}]
result = call_llm_failover(test_msg, max_tokens=10)
if result.get("success"):
    print(f"Connexion OK. Modele : {result['model']} ({result['latency_ms']} ms)")
else:
    print("Echec de connexion :", result.get("error"))

Connexion OK. Modele : openai/gpt-oss-20b:free (13961 ms)


## 4. Patient Profile

In [40]:
PATIENT = {
    "glucose": 148, "blood_pressure": 72, "skin_thickness": 35, "insulin": 0,
    "bmi": 33.6, "diabetes_pedigree_function": 0.627, "age": 50, "pregnancies": 1,
    "mlp_probability": 0.78, "mlp_prediction": "Diabetique", "mlp_model_version": "mlp_v3",
    "hf_top_category": "diabetes symptoms", "hf_confidence": 0.87, "hf_model": "MoritzLaurer/deberta-v3-large-zeroshot-v2.0",
    "symptoms_text": "I feel very thirsty all the time, I urinate frequently, I am always tired and my vision is getting blurry."
}

## 5. Prompt v1

In [41]:
prompt_v1 = f"""A patient has a {int(PATIENT['mlp_probability']*100)}% diabetes risk score.
Their symptoms: {PATIENT['symptoms_text']}
Provide a brief medical summary in 2-3 sentences."""

res_v1 = call_llm_failover([{"role": "user", "content": prompt_v1}], max_tokens=300)
print(f"Modele: {res_v1.get('model')}")
print(res_v1.get('content'))

Modele: openai/gpt-oss-20b:free
The patient’s high diabetes risk score (78%) coupled with classic hyperglycemic symptoms—polydipsia, polyuria, fatigue, and progressive blurred vision—suggests undiagnosed or poorly controlled type 2 diabetes mellitus. Immediate fasting glucose or HbA1c testing, along with a comprehensive metabolic panel, is warranted to confirm the diagnosis and initiate appropriate glycemic management.


## 6. Prompt v2

In [42]:
sys_v2 = "You are a medical AI assistant. Return ONLY valid JSON."
usr_v2 = f"""Analyze this patient profile and return a JSON report.
PATIENT DATA:
- Risk score: {int(PATIENT['mlp_probability']*100)}%
- Symptoms: {PATIENT['symptoms_text']}
- HF Classification: {PATIENT['hf_top_category']} (confidence: {PATIENT['hf_confidence']:.0%})
- Glucose: {PATIENT['glucose']}, BMI: {PATIENT['bmi']}, Age: {PATIENT['age']}

Return ONLY JSON:
{{
  "risk_level": "low|moderate|high",
  "summary": "2 sentences",
  "recommendations": ["rec1", "rec2"],
  "urgency": "routine|soon|immediate"
}}"""

raw_v2 = call_llm_failover([{"role": "system", "content": sys_v2}, {"role": "user", "content": usr_v2}])
parsed_v2 = extract_json(raw_v2.get('content', '')) or local_fallback(PATIENT)
print(json.dumps(parsed_v2, indent=2))

{
  "risk_level": "high",
  "summary": "The patient exhibits classic hyperglycemic symptoms\u2014polydipsia, polyuria, fatigue, and blurred vision\u2014alongside a BMI of 33.6 and an elevated glucose of 148 mg/dL, indicating a high likelihood of type 2 diabetes. With a risk score of 78% and strong clinical indicators, prompt evaluation is essential to prevent complications.",
  "recommendations": [
    "Schedule an urgent fasting plasma glucose and HbA1c test",
    "Consult an endocrinologist for diabetes management and lifestyle counseling"
  ],
  "urgency": "immediate"
}


## 7. Prompt v3 - Chain of Thought

In [43]:
sys_v3 = "You are a medical AI assistant. Reason step by step, but output ONLY the JSON."
usr_v3 = f"""Patient Assessment:
[MLP Risk]: {int(PATIENT['mlp_probability']*100)}%
[HF Category]: {PATIENT['hf_top_category']}
[Clinical]: Glucose={PATIENT['glucose']}, BMI={PATIENT['bmi']}
[Symptoms]: {PATIENT['symptoms_text']}

Steps: 1. Evaluate tabular risk 2. Check symptoms 3. Final risk

Output ONLY this JSON:
{{
  "risk_level": "low|moderate|high",
  "summary": "3 sentences",
  "key_risk_factors": ["f1", "f2"],
  "recommendations": ["a1", "a2"],
  "urgency": "routine|soon|immediate"
}}"""

raw_v3 = call_llm_failover([{"role": "system", "content": sys_v3}, {"role": "user", "content": usr_v3}], max_tokens=800)
parsed_v3 = extract_json(raw_v3.get('content', '')) or local_fallback(PATIENT)
print(json.dumps(parsed_v3, indent=2))

{
  "risk_level": "high",
  "summary": "The patient exhibits classic hyperglycemic symptoms\u2014persistent thirst, frequent urination, fatigue, and blurry vision\u2014alongside a fasting glucose of 148 mg/dL and a BMI of 33.6, indicating obesity. The MLP risk score of 78% further confirms a high probability of undiagnosed diabetes or pre\u2011diabetes. Immediate evaluation and management are essential to prevent acute complications and long\u2011term organ damage.",
  "key_risk_factors": [
    "elevated fasting glucose (148 mg/dL)",
    "obesity (BMI 33.6)",
    "polyuria, polydipsia, fatigue, blurry vision"
  ],
  "recommendations": [
    "Obtain an HbA1c test and repeat fasting glucose to confirm diagnosis",
    "Refer to an endocrinologist for comprehensive diabetes management",
    "Initiate lifestyle modifications: diet plan, weight\u2011loss program, and regular physical activity",
    "Educate on monitoring blood glucose, recognizing hypoglycemia, and proper foot care",
    "Sc

## 8. Compare

In [44]:
comps = [
    {"Version": "v1", "Model": res_v1.get("model", "error")[:25], "OK": res_v1.get("success", False), "Lat": res_v1.get("latency_ms", 0)},
    {"Version": "v2", "Model": raw_v2.get("model", "error")[:25], "OK": raw_v2.get("success", False), "Lat": raw_v2.get("latency_ms", 0)},
    {"Version": "v3", "Model": raw_v3.get("model", "error")[:25], "OK": raw_v3.get("success", False), "Lat": raw_v3.get("latency_ms", 0)},
]
print(pd.DataFrame(comps).to_string(index=False))

Version                   Model   OK   Lat
     v1 openai/gpt-oss-20b:free True 16684
     v2 openai/gpt-oss-20b:free True 20952
     v3 openai/gpt-oss-20b:free True 21325


## 9. Full pipeline & Save

In [45]:
pipe = {
    "patient_input": {
        "glucose": PATIENT["glucose"],
        "blood_pressure": PATIENT["blood_pressure"],
        "skin_thickness": PATIENT["skin_thickness"],
        "insulin": PATIENT["insulin"],
        "bmi": PATIENT["bmi"],
        "diabetes_pedigree_function": PATIENT["diabetes_pedigree_function"],
        "age": PATIENT["age"],
        "pregnancies": PATIENT["pregnancies"],
        "symptoms_text": PATIENT["symptoms_text"],
    },
    "results": {
        "classic_model": {
            "probability": PATIENT["mlp_probability"],
            "prediction": PATIENT["mlp_prediction"],
            "model_version": PATIENT["mlp_model_version"],
        },
        "huggingface": {
            "top_category": PATIENT["hf_top_category"],
            "confidence": PATIENT["hf_confidence"],
            "model": PATIENT["hf_model"],
        },
        "llm": {
            "report": parsed_v3,
            "model_used": raw_v3.get("model"),
            "latency_ms": raw_v3.get("latency_ms"),
            "prompt_version": "v3_chain_of_thought",
        },
    },
}
with open(MODELS_DIR / "pipeline_simulation.json", "w") as f: json.dump(pipe, f, indent=2)

best_res = {"model": raw_v3.get("model", "local"), "parsed_json": parsed_v3, "latency_ms": raw_v3.get("latency_ms", 0)}
with open(MODELS_DIR / "llm_best_result.json", "w") as f: json.dump(best_res, f, indent=2)
print("Pipeline et Best Result saved.")

Pipeline et Best Result saved.
